In [1]:
from kafka import KafkaConsumer, TopicPartition
import json
import os, logging
from datetime import datetime, timedelta
from baskervillehall.model_io import ModelIO

In [2]:
# kafka_url = ['kafka9-0.kafka9-headless.default.svc.cluster.local:9093','kafka9-1.kafka9-headless.default.svc.cluster.local:9093','kafka9-2.kafka9-headless.default.svc.cluster.local:9093']
kafka_url = ['kafkab-0.kafkab-headless.default.svc.cluster.local:9093','kafkab-1.kafkab-headless.default.svc.cluster.local:9093','kafkab-2.kafkab-headless.default.svc.cluster.local:9093']

# topic = 'logstash_deflect.log'
topic = 'banjax_command_topic'
datetime_format = '%Y-%m-%d %H:%M:%S'
partitions = {
    'zhitomir.info': 1,
    'urban-pushkino.ru': 0,
    'dev.emawpb.org': 0,
    'palestinechronicle.com': 1,
    'equalit.ie': 0,
    'lexota.org': 0,
    'kavkaz-uzel.eu': 0,
    'amp.kavkaz-uzel.eu': 2,
    'indymedia.nl': 0,
    'verafiles.org': 1,
    'anticor-kharkiv.org': 0,
}

In [3]:
logger = logging.getLogger('pca')
logger.addHandler(logging.StreamHandler())
logger.setLevel('DEBUG')
os.environ['S3_ACCESS'] = 'c490f17bdb784fa08f4d11836ee18e48'
os.environ['S3_SECRET'] = 'e4b65e27b3734d0d96ce6038586ef43f'
os.environ['S3_ENDPOINT'] = 's3.gra.cloud.ovh.net'
os.environ['S3_REGION'] = 'GRA'
s3_connection = {
    's3_access':os.environ['S3_ACCESS'],
    's3_secret':os.environ['S3_SECRET'],
    's3_endpoint': os.environ['S3_ENDPOINT'],
    's3_region':os.environ['S3_REGION']
}
model_io = ModelIO(**s3_connection, logger=logger)

In [23]:
def read_dataset(host, partition, start, stop, datetime_format, kafka_url, topic):
    consumer = KafkaConsumer(
        bootstrap_servers=kafka_url,
        group_id='logs2'
    )
    print(f'Reading from kafka. Host = {host} ... partition = {partition}')
    num = 0
    logs = []
    ips = set()

    consumer.assign([TopicPartition(topic, partition)])
    consumer.seek_to_beginning()
    # consumer.seek(TopicPartition(topic, partition), 444580)
    
    complete = False
    read = 0
    while not complete:
        raw_messages = consumer.poll(timeout_ms=1000, max_records=5000)

        for topic_partition, messages in raw_messages.items():
            for message in messages:
                if message.value is None :
                    continue
                if message.key is None:
                    continue
                if message.key.decode("utf-8") != host:
                    continue 
                value = json.loads(message.value.decode("utf-8"))
                # ts = datetime.strptime(value['datestamp'], '%Y-%m-%dT%H:%M:%S.%fZ')
                ts = value['end']

                if read % 1000 == 0 or read == 0:
                    print(ts)
                read += 1
                # if ts > stop:
                #     complete = True
                #     break

                if value['Value'] == '35.243.23.69':
                    import pdb
                    pdb.set_trace()
                    
                # if ts > start and ts < stop:
                #     ips.add(value['client_ip']) 
                #     logs.append(value)
                #     num += 1
                #     if num % 10000 == 0:
                #         print(f'{num} logs read', ts, message.timestamp)

            
    return ips, logs

In [24]:
host = 'verafiles.org'
start = datetime(2024, 1, 4, 9, 50, 0, 0)
stop = datetime(2024, 1, 4, 9, 54, 50, 0)

In [ ]:
ips, logs = read_dataset(
    host, partitions[host], start, stop, datetime_format, 
    kafka_url='kafkab-0.kafkab-headless.default.svc.cluster.local:9093,kafkab-2.kafkab-headless.default.svc.cluster.local:9093,kafkab-2.kafkab-headless.default.svc.cluster.local:9093',
    topic=topic
)
                            

Reading from kafka. Host = verafiles.org ... partition = 1
2024-05-08 18:18:56
2024-05-09 02:33:32
2024-05-09 06:51:01
2024-05-09 12:47:02
2024-05-09 16:06:07
2024-05-10 01:40:00
2024-05-10 09:10:21
2024-05-10 15:44:59
2024-05-11 00:10:05
2024-05-11 07:47:38
2024-05-11 16:04:17
2024-05-11 23:51:18
2024-05-12 10:22:44
2024-05-12 17:48:55
2024-05-13 02:20:46
2024-05-13 07:40:17
2024-05-13 11:34:30
2024-05-13 16:00:01
2024-05-13 21:15:15
2024-05-14 03:11:32
2024-05-14 05:25:54
2024-05-14 08:06:33
2024-05-14 10:36:50
2024-05-14 13:09:34
2024-05-14 14:26:34
2024-05-14 16:09:39
2024-05-14 18:34:31
2024-05-14 20:42:09
2024-05-14 22:21:53
2024-05-15 00:28:06
2024-05-15 02:14:57
2024-05-15 04:19:39
2024-05-15 07:04:05
2024-05-15 08:03:50
2024-05-15 09:30:54
2024-05-15 10:51:20
2024-05-15 12:19:05
2024-05-15 13:46:08
2024-05-15 15:21:28
2024-05-15 16:59:03
2024-05-15 18:39:36
2024-05-15 21:24:27
2024-05-15 23:14:05
2024-05-16 00:46:00
2024-05-16 02:22:42
2024-05-16 04:10:02
2024-05-16 05:47:00
2

ipdb>  w


    [... skipping 21 hidden frame(s)]

  /tmp/ipykernel_131/3659509921.py(1)<module>()
----> 1 ips, logs = read_dataset(
      2     host, partitions[host], start, stop, datetime_format,
      3     kafka_url='kafkab-0.kafkab-headless.default.svc.cluster.local:9093,kafkab-2.kafkab-headless.default.svc.cluster.local:9093,kafkab-2.kafkab-headless.default.svc.cluster.local:9093',
      4     topic=topic
      5 )

> /tmp/ipykernel_131/2611479596.py(21)read_dataset()
     19 
     20         for topic_partition, messages in raw_messages.items():
---> 21             for message in messages:
     22                 if message.value is None :
     23                     continue



ipdb>  message


ConsumerRecord(topic='banjax_command_topic', partition=1, offset=1021029, timestamp=1716171536069, timestamp_type=0, key=b'verafiles.org', value=b'{"duration":92.0,"session_id":"-","end":"2024-05-20 02:18:52","start":"2024-05-20 02:17:20","score":-0.13259826359111127,"num_requests":11,"host":"verafiles.org","source":"baskervillehall","Value":"35.243.23.69","Name":"challenge_ip"}', headers=[], checksum=None, serialized_key_size=13, serialized_value_size=234, serialized_header_size=-1)


In [36]:
ips, logs = read_dataset(host, partitions[host], start, stop, datetime_format, kafka_url)
print(f'{len(logs)} logs, {len(ips)} ips')

Reading from kafka. Host = verafiles.org ... partition = 1
2024-01-04 09:52:45
2024-01-04 09:52:48
10000 logs read 2024-01-04 09:52:04 1704361969959
2024-01-04 09:52:49
20000 logs read 2024-01-04 09:52:06 1704361971902
2024-01-04 09:52:51
30000 logs read 2024-01-04 09:52:19 1704361972799
2024-01-04 09:52:52
40000 logs read 2024-01-04 09:52:03 1704361973512
2024-01-04 09:52:52
50000 logs read 2024-01-04 09:52:52 1704361974348
2024-01-04 09:52:44
60000 logs read 2024-01-04 09:52:44 1704361975172
2024-01-04 09:52:54
70000 logs read 2024-01-04 09:52:53 1704361976107
2024-01-04 09:52:53
80000 logs read 2024-01-04 09:52:54 1704361976875
2024-01-04 09:52:53
90000 logs read 2024-01-04 09:52:54 1704361977698
2024-01-04 09:52:54
100000 logs read 2024-01-04 09:52:54 1704361978174
2024-01-04 09:52:54
110000 logs read 2024-01-04 09:52:44 1704361978873
2024-01-04 09:52:44
120000 logs read 2024-01-04 09:52:44 1704361979186
2024-01-04 09:52:55
130000 logs read 2024-01-04 09:52:57 1704361979993
2024-01

In [19]:
model_io._save_object(json.dumps(logs, indent=2), 'anton/dataset/logs/verafiles_logs_2024_01_04.json')

In [52]:
len(ips)

968

In [53]:
print(len(set([s['client_ip'] for s in logs])))

968


In [54]:
print(set([s['deflect_session'] for s in logs]))

{'', '3CIGh7%2FkD30AAAAAZZaMDA%3D%3D', '1j3rrYCHp1kAAAAAZZaM9g%3D%3D', 'VlAg3lqJXE4AAAAAZZaB3A%3D%3D', 'bsaJA3%2B6J1sAAAAAZZaMeg%3D%3D', '-', 'G5bSgKWtD4QAAAAAZZaMzA%3D%3D', 'IyFilFm3D1IAAAAAZZaLeg%3D%3D', 'KwGnqjHbr8IAAAAAZZaL0A%3D%3D'}


In [58]:
for log in logs:
    if log['client_ip'] == '190.2.210.116':
        print(log['datestamp'])

2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z
2024-01-04T09:52:53.000Z


In [21]:
from kafka import KafkaProducer

In [41]:
producer = KafkaProducer(bootstrap_servers='kafkab-0.kafkab-headless.default.svc.cluster.local:9093,kafkab-2.kafkab-headless.default.svc.cluster.local:9093,kafkab-2.kafkab-headless.default.svc.cluster.local:9093')

In [23]:
for log in logs:
    producer.send(
        'anton_logs',
        key=bytearray(host, encoding='utf8'),
        value=json.dumps(log).encode('utf-8')
    )